# Imports

In [21]:
import torch
from model import NestedUNet
from tqdm import tqdm
from metrics import iou_score
from collections import OrderedDict
from utils import AverageMeter, str2bool
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.optim as optim
import losses
import pandas as pd
import random
import numpy as np

# Model checking

In [ ]:
# num_classes = 10      # segmentation with 3 classes (background + 2 organs/tumors for example)
# input_channels = 3   # RGB input
# batch_size = 2        # just for testing
# H, W = 256, 256       # input image size

# # Instantiate the model
# model = NestedUNet(num_classes=num_classes, input_channels=input_channels, deep_supervision=False)

# # Random input tensor
# x = torch.randn(batch_size, input_channels, H, W)

# # Forward pass
# out = model(x)

# print("Output shape (no deep supervision):", out.shape)


Output shape (no deep supervision): torch.Size([2, 10, 256, 256])


In [6]:
# print(model)

In [ ]:
# # Instantiate with deep supervision
# model_ds = NestedUNet(num_classes=num_classes, input_channels=input_channels, deep_supervision=True)

# # Forward pass
# outs = model_ds(x)

# print("Number of outputs (deep supervision):", len(outs))
# for i, o in enumerate(outs, 1):
#     print(f"Output {i} shape:", o.shape)


Number of outputs (deep supervision): 4
Output 1 shape: torch.Size([2, 10, 256, 256])
Output 2 shape: torch.Size([2, 10, 256, 256])
Output 3 shape: torch.Size([2, 10, 256, 256])
Output 4 shape: torch.Size([2, 10, 256, 256])


In [ ]:
# # Total parameters
# total_params = sum(p.numel() for p in model.parameters())
# # Trainable parameters
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# print(f"Total parameters: {total_params:,}")
# print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 9,163,626
Trainable parameters: 9,163,626


# seeds

In [22]:
seed = 42  # you can choose any integer
random.seed(seed)                   # Python RNG
np.random.seed(seed)                # Numpy RNG
torch.manual_seed(seed)             # PyTorch RNG (CPU)
torch.cuda.manual_seed(seed)        # PyTorch RNG (current GPU)
torch.cuda.manual_seed_all(seed)    # PyTorch RNG (all GPUs)

# For deterministic behavior (slower but reproducible)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import random

seed = 42


random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)  # optional
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


In [ ]:
import torchvision.transforms.functional as F
from torchvision.transforms import functional as TF

In [ ]:

def collate_fn(batch):
    def to_tensor(img):
        if isinstance(img, torch.Tensor):
            return img  # already tensor
        elif img.ndim == 2:
            return torch.from_numpy(img).unsqueeze(0)
        else:
            return torch.from_numpy(img).permute(2, 0, 1)
    images   = [to_tensor(sample["image"]).float()   for sample in batch]
    semantics = [torch.from_numpy(sample["semantic"]).long()
                 for sample in batch]

    return {
        "image":    torch.stack(images),      # B × C × H × W
        "semantic": torch.stack(semantics)    # B × H × W
    }  

In [ ]:
import logging
import albumentations as A
from detectron2.config import get_cfg, CfgNode
from detectron2.utils.events import _CURRENT_STORAGE_STACK, EventStorage
import sys
# from EfficientPS.datasets.panoptic_dataset import PanopticDataset, collate_fn

In [ ]:
def add_custom_param(cfg):
    """
    In order to add custom config parameter in the .yaml those parameter must
    be initialised
    """
    # Model
    cfg.MODEL_CUSTOM = CfgNode()
    cfg.MODEL_CUSTOM.BACKBONE = CfgNode()
    cfg.MODEL_CUSTOM.BACKBONE.EFFICIENTNET_ID = 7
    cfg.MODEL_CUSTOM.BACKBONE.LOAD_PRETRAIN = True
    # DATASET
    cfg.NUM_CLASS = 2
    cfg.DATASET_PATH = "/mnt/e3dbc9b9-6856-470d-84b1-ff55921cd906/EPS_Medical/Atlas_dataset"
    cfg.TRAIN_JSON = "/mnt/e3dbc9b9-6856-470d-84b1-ff55921cd906/EPS_Medical/Liver_dataset/output/cityscapes_panoptic_train.json"
    cfg.VALID_JSON = "/mnt/e3dbc9b9-6856-470d-84b1-ff55921cd906/EPS_Medical/Liver_dataset/output/cityscapes_panoptic_val.json"
    cfg.PRED_DIR = "/mnt/e3dbc9b9-6856-470d-84b1-ff55921cd906/EPS_Medical/Liver_dataset/preds"
    cfg.PRED_JSON = "/mnt/e3dbc9b9-6856-470d-84b1-ff55921cd906/EPS_Medical/Liver_dataset/preds/cityscapes_panoptic_preds.json"
    # Transfom
    cfg.TRANSFORM = CfgNode()
    cfg.TRANSFORM.NORMALIZE = CfgNode()
    cfg.TRANSFORM.NORMALIZE.MEAN = (0.5,)
    cfg.TRANSFORM.NORMALIZE.STD  = (0.5,)
    cfg.TRANSFORM.RESIZE = CfgNode()
    cfg.TRANSFORM.RESIZE.HEIGHT = 256
    cfg.TRANSFORM.RESIZE.WIDTH = 256
    cfg.TRANSFORM.RANDOMCROP = CfgNode()
    cfg.TRANSFORM.RANDOMCROP.HEIGHT = 256
    cfg.TRANSFORM.RANDOMCROP.WIDTH = 256
    cfg.TRANSFORM.HFLIP = CfgNode()
    cfg.TRANSFORM.HFLIP.PROB = 0.5
    # Solver
    cfg.SOLVER.NAME = "AdamW"
    cfg.SOLVER.ACCUMULATE_GRAD = 1
    cfg.SOLVER.MAX_EPOCHS = 100
    # Runner
    cfg.BATCH_SIZE = 8
    cfg.CHECKPOINT_PATH = ""
    cfg.PRECISION = 32
    # Callbacks
    cfg.CALLBACKS = CfgNode()
    cfg.CALLBACKS.CHECKPOINT_DIR = None
    # Inference
    cfg.INFERENCE = CfgNode()
    cfg.INFERENCE.AREA_TRESH = 0


In [ ]:

from torch.utils.data import Dataset

class PanopticcDataset(Dataset):
    """
    Minimal dataset class that now takes pre-loaded image/label data lists.
    """

    def __init__(self, image_list, label_list, transform=None):
        """
        Args:
            image_list (List[np.ndarray]): List of image arrays.
            label_list (List[np.ndarray]): List of label arrays.
            transform (callable, optional): Transform to apply (e.g. torchvision).
        """
        self.image_data = image_list
        self.label_data = label_list
        self.transform = transform

        if len(self.image_data) != len(self.label_data):
            raise ValueError("Image and label list lengths do not match.")

    def __len__(self):
        return len(self.image_data)

    def __getitem__(self, idx):
        image = self.image_data[idx]
        semantic = self.label_data[idx]

        if image is None or semantic is None:
            raise RuntimeError(f"Failed to load image or label at index {idx}")

        # --- optional transform ----------------------------------------
        # if self.transform is not None:
        #     image_pil = Image.fromarray(image, mode='L')
        #     image = self.transform(image_pil)  # returns torch.Tensor
        #     # semantic remains as NumPy (or you can also convert it to torch.Tensor here)

        if self.transform is not None:
            # image_pil = Image.fromarray(image, mode='L')
            # image = self.transform(image_pil)  # result is a torch.Tensor
            transformed = self.transform(image=image, mask=semantic, seed=42)
            image = transformed["image"]
            semantic = transformed["mask"]

        return {
            "image": image,
            "semantic": semantic
        }

In [ ]:
import torch
import torchvision.transforms as T
import numpy as np


import os
import random
import numpy as np
import torch

SEED = 42

# Set environment-level seeds
# os.environ["PYTHONHASHSEED"] = str(SEED)

# Python, NumPy, and PyTorch seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # For multi-GPU setups

# Force deterministic operations
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



cfg = get_cfg()
add_custom_param(cfg)
cfg.merge_from_file("/mnt/e3dbc9b9-6856-470d-84b1-ff55921cd906/EPS_Medical/EfficientPS/config.yaml")

logging.getLogger("pytorch_lightning").setLevel(logging.INFO)
logger = logging.getLogger("pytorch_lightning.core")
if not os.path.exists(cfg.CALLBACKS.CHECKPOINT_DIR):
    os.makedirs(cfg.CALLBACKS.CHECKPOINT_DIR)
logger.addHandler(logging.FileHandler(
    os.path.join(cfg.CALLBACKS.CHECKPOINT_DIR,"core.log")))
with open("/mnt/e3dbc9b9-6856-470d-84b1-ff55921cd906/EPS_Medical/EfficientPS/config.yaml") as file:
    logger.info(file.read())
# Initialise Custom storage to avoid error when using detectron 2
_CURRENT_STORAGE_STACK.append(EventStorage())



transform_train = A.Compose([
    A.Resize(
        height=cfg.TRANSFORM.RESIZE.HEIGHT,
        width=cfg.TRANSFORM.RESIZE.WIDTH
    ),
    A.RandomCrop(
        height=cfg.TRANSFORM.RANDOMCROP.HEIGHT,
        width=cfg.TRANSFORM.RANDOMCROP.WIDTH
    ),
    A.HorizontalFlip(p=cfg.TRANSFORM.HFLIP.PROB),
    A.Normalize(
        mean=cfg.TRANSFORM.NORMALIZE.MEAN,
        std=cfg.TRANSFORM.NORMALIZE.STD
    ),
], seed=42)

transform_valid = A.Compose([
    A.Resize(height=256, width=256),
    A.Normalize(
        mean=cfg.TRANSFORM.NORMALIZE.MEAN,
        std=cfg.TRANSFORM.NORMALIZE.STD
    ),
], seed=42)


In [ ]:
import random
import numpy as np
import torch

SEED = 42

# Set environment-level seeds
# os.environ["PYTHONHASHSEED"] = str(SEED)

# Python, NumPy, and PyTorch seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # For multi-GPU setups

# Force deterministic operations
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from torch.utils.data import DataLoader
import torch



def worker_init_fn(worker_id):
    np.random.seed(42 + worker_id)


train_dataset = PanopticcDataset(
    path_json=None,
    root_dir=cfg.DATASET_PATH,   # e.g. "/…/Atlas_dataset"
    split="train",
    transform=transform_train
)

valid_dataset = PanopticcDataset(
    path_json=None,
    root_dir=cfg.DATASET_PATH,
    split="val",
    transform=transform_valid
)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn,
    worker_init_fn=worker_init_fn,
    # generator=g,
    pin_memory=False
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
    worker_init_fn=worker_init_fn,
    # generator=g,
    pin_memory=False
)


# Training code

In [ ]:
def train(deep_supervision, train_loader, model, criterion, optimizer):
    avg_meters = {'loss': AverageMeter(),
                  'iou': AverageMeter()}

    model.train()

    pbar = tqdm(total=len(train_loader))
    for input, target, _ in train_loader:
        input = input.cuda()
        target = target.cuda()

        # compute output
        if deep_supervision:
            outputs = model(input)
            loss = 0
            for output in outputs:
                loss += criterion(output, target)
            loss /= len(outputs)
            iou = iou_score(outputs[-1], target)
        else:
            output = model(input)
            loss = criterion(output, target)
            iou = iou_score(output, target)

        # compute gradient and do optimizing step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        avg_meters['loss'].update(loss.item(), input.size(0))
        avg_meters['iou'].update(iou, input.size(0))

        postfix = OrderedDict([
            ('loss', avg_meters['loss'].avg),
            ('iou', avg_meters['iou'].avg),
        ])
        pbar.set_postfix(postfix)
        pbar.update(1)
    pbar.close()

    return OrderedDict([('loss', avg_meters['loss'].avg),
                        ('iou', avg_meters['iou'].avg)])

In [ ]:
def validate(deep_supervision, val_loader, model, criterion):
    avg_meters = {'loss': AverageMeter(),
                  'iou': AverageMeter()}

    # switch to evaluate mode
    model.eval()

    with torch.no_grad():
        pbar = tqdm(total=len(val_loader))
        for input, target, _ in val_loader:
            input = input.cuda()
            target = target.cuda()

            # compute output
            if deep_supervision:
                outputs = model(input)
                loss = 0
                for output in outputs:
                    loss += criterion(output, target)
                loss /= len(outputs)
                iou = iou_score(outputs[-1], target)
            else:
                output = model(input)
                loss = criterion(output, target)
                iou = iou_score(output, target)

            avg_meters['loss'].update(loss.item(), input.size(0))
            avg_meters['iou'].update(iou, input.size(0))

            postfix = OrderedDict([
                ('loss', avg_meters['loss'].avg),
                ('iou', avg_meters['iou'].avg),
            ])
            pbar.set_postfix(postfix)
            pbar.update(1)
        pbar.close()

    return OrderedDict([('loss', avg_meters['loss'].avg),
                        ('iou', avg_meters['iou'].avg)])

In [ ]:
num_epochs = 10
deep_supervision = True
scheduler = "CosineAnnealingLR"  # ReduceLROnPlateau
loss = "BCEWithLogitsLoss"  # LovaszHingeLoss # BCEDiceLoss
log = OrderedDict([
        ('epoch', []),
        ('lr', []),
        ('loss', []),
        ('iou', []),
        ('val_loss', []),
        ('val_iou', []),
    ])
name = "ram"                     # model name: default = arch+timestamp
batch_size = 16                 # mini-batch size (default: 16)
arch = "NestedUNet"      # model architecture: NestedUNet | UNet | etc.
deep_supervision = False # True if using deep supervision
input_channels = 3       # number of input channels
num_classes = 1          # number of output classes
input_w = 96             # image width
input_h = 96             # image height
scheduler = "CosineAnnealingLR"   # Options: CosineAnnealingLR | ReduceLROnPlateau | MultiStepLR | ConstantLR
min_lr = 1e-5                     # minimum learning rate
factor = 0.1                      # factor for ReduceLROnPlateau
patience = 2                      # patience for ReduceLROnPlateau
milestones = [1, 2]               # milestones for MultiStepLR
gamma = 2/3                       # gamma for MultiStepLR
early_stopping = -1               # early stopping (default: -1 means disabled)
cudnn.benchmark = True

In [ ]:
model = NestedUNet(num_classes=num_classes, input_channels=input_channels, deep_supervision=deep_supervision)

In [19]:
if loss == 'BCEWithLogitsLoss':
    criterion = nn.BCEWithLogitsLoss().cuda()
else:
    criterion = losses.__dict__[loss]().cuda()

optimizer_type = "SGD"       # "Adam" or "SGD"
learning_rate = 1e-3         # default: 0.001
momentum = 0.9               # only used if optimizer == SGD
weight_decay = 1e-4          # L2 regularization
nesterov = False             # True/False, only relevant for SGD
if optimizer_type == "Adam":
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
elif optimizer_type == "SGD":
    optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                          momentum=momentum, weight_decay=weight_decay,
                          nesterov=nesterov)

In [ ]:
best_iou = 0
trigger = 0
for epoch in range(num_epochs):
    print('Epoch [%d/%d]' % (epoch, num_epochs))

    # train for one epoch
    train_log = train(deep_supervision, train_loader, model, criterion, optimizer)
    # evaluate on validation set
    val_log = validate(deep_supervision, val_loader, model, criterion)

    if scheduler == 'CosineAnnealingLR':
        scheduler.step()
    elif scheduler == 'ReduceLROnPlateau':
        scheduler.step(val_log['loss'])

    print('loss %.4f - iou %.4f - val_loss %.4f - val_iou %.4f'
            % (train_log['loss'], train_log['iou'], val_log['loss'], val_log['iou']))

    log['epoch'].append(epoch)
    log['lr'].append(learning_rate)
    log['loss'].append(train_log['loss'])
    log['iou'].append(train_log['iou'])
    log['val_loss'].append(val_log['loss'])
    log['val_iou'].append(val_log['iou'])

    # pd.DataFrame(log).to_csv('models/%s/log.csv' %
    #                             config['name'], index=False)

    trigger += 1

    if val_log['iou'] > best_iou:
        torch.save(model.state_dict(), 'models/%s/model.pth' %
                    name)
        best_iou = val_log['iou']
        print("=> saved best model")
        trigger = 0

    # early stopping
    if early_stopping >= 0 and trigger >= early_stopping:
        print("=> early stopping")
        break

    torch.cuda.empty_cache()